# Creative Performance Scoring Pipeline

This notebook is an anonymised portfolio version of a workplace analytics workflow. It calculates a weighted performance score for marketing creatives running in long-lived campaign environments.

No proprietary data, internal table names, credentials, dashboard links, or confidential business logic are included.

## Project context

The purpose of this workflow is to help marketing and creative teams compare creative assets using a consistent scoring methodology.

The score combines three components:

| Component | Description |
|---|---|
| **Volume** | Measures a creative's share of impressions and installs within its campaign. |
| **Experience** | Measures how long the creative has been active compared with the oldest creative in the same campaign. |
| **Value** | Measures the creative's share of total campaign value. |

In [ ]:
# Standard imports
from datetime import datetime

import numpy as np
import pandas as pd

# BigQuery client
from google.cloud import bigquery

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

## Configuration

All project IDs, dataset names, and table names below are placeholders for portfolio use.

In [ ]:
GCP_PROJECT_ID = "your-gcp-project-id"
BQ_DATASET_ID = "analytics_dataset"
BQ_OUTPUT_TABLE = "creative_performance_scores"

REPORTING_INTERVAL_DAYS = 90

VOLUME_WEIGHT = 0.5
EXPERIENCE_WEIGHT = 0.3
VALUE_WEIGHT = 0.2

THRESHOLDS_INSTALLS = {
    "Network A": 100,
    "Network B": 150,
    "Network C": 200,
}

client = bigquery.Client(project=GCP_PROJECT_ID)

## Get campaign creative data

The production version read from internal marketing performance tables. This anonymised version reads from a portfolio-safe SQL file using placeholder table names.

In [ ]:
with open("../sql/get_campaign_creative_data.sql", "r") as file:
    query = file.read().replace("@reporting_interval_days", str(REPORTING_INTERVAL_DAYS))

query_job = client.query(query)
campaigns_creatives = query_job.result().to_dataframe()

In [ ]:
campaigns_creatives = (
    campaigns_creatives
    .copy(deep=True)
    .astype({
        "network": "object",
        "product_name": "object",
        "campaign_name": "object",
        "creative_name": "object",
        "first_in_campaign_install_date": "datetime64[ns]",
        "last_in_campaign_install_date": "datetime64[ns]",
        "tot_installs": "int64",
        "tot_impressions": "int64",
        "tot_cost": "float64",
        "tot_value": "float64"
    })
)

campaigns_creatives.head()

# Component calculations

## Volume score

The volume score measures each creative's share of total campaign impressions and installs. Both shares are multiplied by 10 to make the score easier to read.

In [ ]:
campaigns_creatives["tot_impressions_campaign"] = (
    campaigns_creatives
    .groupby("campaign_name")["tot_impressions"]
    .transform("sum")
)

campaigns_creatives["tot_installs_campaign"] = (
    campaigns_creatives
    .groupby("campaign_name")["tot_installs"]
    .transform("sum")
)

campaigns_creatives = campaigns_creatives.loc[
    (campaigns_creatives["tot_impressions_campaign"] > 0)
    & (campaigns_creatives["tot_installs_campaign"] > 0),
    :
]

campaigns_creatives["impressions_share_x10"] = (
    campaigns_creatives["tot_impressions"]
    / campaigns_creatives["tot_impressions_campaign"]
) * 10

campaigns_creatives["installs_share_x10"] = (
    campaigns_creatives["tot_installs"]
    / campaigns_creatives["tot_installs_campaign"]
) * 10

campaigns_creatives["volume_score"] = (
    campaigns_creatives[["impressions_share_x10", "installs_share_x10"]]
    .mean(axis=1)
)

## Experience score

The experience score measures how long a creative has been active compared with the oldest creative in the same campaign.

In [ ]:
campaigns_creatives["age"] = (
    campaigns_creatives["last_in_campaign_install_date"]
    - campaigns_creatives["first_in_campaign_install_date"]
).dt.days

campaigns_creatives["oldest_creative_in_campaign_age"] = (
    campaigns_creatives
    .groupby("campaign_name")["age"]
    .transform("max")
)

campaigns_creatives["experience_score"] = np.where(
    campaigns_creatives["oldest_creative_in_campaign_age"] > 0,
    (
        campaigns_creatives["age"]
        / campaigns_creatives["oldest_creative_in_campaign_age"]
    ) * 10,
    0
)

## Value score

The value score measures each creative's share of total campaign value. In this portfolio version, value is represented by a generic `tot_value` field.

In [ ]:
campaigns_creatives["tot_value_campaign"] = (
    campaigns_creatives
    .groupby("campaign_name")["tot_value"]
    .transform("sum")
)

campaigns_creatives["value_score"] = np.where(
    campaigns_creatives["tot_value_campaign"] > 0,
    (campaigns_creatives["tot_value"] / campaigns_creatives["tot_value_campaign"]) * 10,
    0
)

# Calculate Creative Fitness Score

The Creative Fitness Score combines the three weighted components into one ranking metric.

```text
Creative Fitness Score =
  (Volume Score × Volume Weight)
+ (Experience Score × Experience Weight)
+ (Value Score × Value Weight)
```

In [ ]:
campaigns_creatives["creative_fitness_score"] = (
    (campaigns_creatives["volume_score"] * VOLUME_WEIGHT)
    + (campaigns_creatives["experience_score"] * EXPERIENCE_WEIGHT)
    + (campaigns_creatives["value_score"] * VALUE_WEIGHT)
)

campaigns_creatives = campaigns_creatives.round(2)

## Install thresholds

Minimum install thresholds help prevent over-interpreting scores for creatives with very limited data.

In [ ]:
campaigns_creatives["threshold_installs"] = (
    campaigns_creatives["network"].map(THRESHOLDS_INSTALLS)
)

campaigns_creatives["threshold_reached"] = np.where(
    campaigns_creatives["tot_installs"] >= campaigns_creatives["threshold_installs"],
    True,
    False
)

## Calculation date

The calculation date is added so the reporting table can store a daily history of scoring outputs.

In [ ]:
campaigns_creatives["calc_date"] = pd.to_datetime(
    datetime.today().strftime("%Y-%m-%d")
)

## Preview scoring output

In [ ]:
campaigns_creatives[[
    "network",
    "product_name",
    "campaign_name",
    "creative_name",
    "volume_score",
    "experience_score",
    "value_score",
    "creative_fitness_score",
    "threshold_reached",
    "calc_date"
]].head()

# Upload to BigQuery

Before uploading, the workflow checks whether today's data already exists. This prevents duplicate daily snapshots.

In [ ]:
BQ_TABLE_ID = f"{GCP_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_OUTPUT_TABLE}"

BQ_TABLE_SCHEMA = [
    bigquery.SchemaField("network", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("product_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("campaign_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("creative_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("first_in_campaign_install_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("last_in_campaign_install_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("tot_installs", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("tot_impressions", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("tot_cost", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("tot_value", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("volume_score", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("experience_score", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("value_score", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("creative_fitness_score", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("threshold_installs", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("threshold_reached", "BOOL", mode="NULLABLE"),
    bigquery.SchemaField("calc_date", "DATE", mode="REQUIRED"),
]

## Check if today's data already exists

In [ ]:
with open("../sql/check_today_upload_exists.sql", "r") as file:
    today_row_count_query = file.read()

today_row_count_response = (
    client.query(today_row_count_query)
    .result()
    .to_dataframe()
)

data_already_exists = bool(today_row_count_response["count_rows"].iloc[0])
data_already_exists

## Upload scored data

The upload is skipped if today's scored data already exists in the destination table.

In [ ]:
if data_already_exists:
    print("Today's data already exists: upload skipped.")
else:
    print("Today's data does not exist: uploading scored data.")

    job_config = bigquery.LoadJobConfig(
        schema=BQ_TABLE_SCHEMA,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )

    load_job = client.load_table_from_dataframe(
        campaigns_creatives,
        BQ_TABLE_ID,
        job_config=job_config,
    )

    load_job.result()
    print("Data successfully uploaded.")

## Anonymisation note

This notebook has been adapted for portfolio use. The following items have been removed or replaced:

- Company names
- Internal project IDs
- Internal dataset and table names
- Internal package names
- Real campaign names
- Real creative names
- Real product names
- Dashboard and scheduler links
- Production data
